# Find First and Latest Order for Each Customer

## Business Scenario

An e-commerce company wants to analyze customer purchase behavior.

For every customer, the business wants to know:

- When was the customer's first order?
- When was the customer's latest order?
- How long has the customer been active?
- Which customers have not placed an order recently?

## Business Use Cases

- ✅ Customer Retention Analysis
- ✅ Customer Lifetime Value (CLV)
- ✅ Identify New vs Loyal Customers
- ✅ Re-engagement Campaigns
- ✅ Churn Analysis

## Sample Data

**orders**

| customer_id | order_id | order_date |
|---|---|---|
| 101 | O1001 | 2026-11-01 |
| 101 | O1002 | 2026-11-05 |
| 101 | O1003 | 2026-11-10 |
| 102 | O1004 | 2026-11-03 |
| 102 | O1005 | 2026-11-15 |
| 103 | O1006 | 2026-11-02 |

## Expected Output

| customer_id | first_order_date | latest_order_date |
|---|---|---|
| 101 | 2026-11-01 | 2026-11-10 |
| 102 | 2026-11-03 | 2026-11-15 |
| 103 | 2026-11-02 | 2026-11-02 |

```sql
SELECT
    customer_id,
    MIN(order_date) AS first_order_date,
    MAX(order_date) AS latest_order_date
FROM orders
GROUP BY customer_id;
```


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

data = [
    (101, "O1001", "2026-11-01", 1200),
    (101, "O1002", "2026-11-05", 800),
    (101, "O1003", "2026-11-10", 1500),

    (102, "O1004", "2026-11-03", 2000),
    (102, "O1005", "2026-11-15", 3500),

    (103, "O1006", "2026-11-02", 900),

    (104, "O1007", "2026-11-01", 500),
    (104, "O1008", "2026-11-07", 1200),
    (104, "O1009", "2026-11-20", 700),

    (105, "O1010", "2026-11-12", 2500)
]

df = spark.createDataFrame(
    data,
    ["customer_id", "order_id", "order_date", "order_amount"]
)

df = df.withColumn(
    "order_date",
    F.to_date("order_date")
)

df.display()

In [0]:
from pyspark.sql import functions as F

result = (
    df.groupBy("customer_id")
      .agg(
          F.min("order_date").alias("first_order_date"),
          F.max("order_date").alias("latest_order_date")
      )
)

result.display()

### Using Window Function??

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = Window.partitionBy("customer_id") \
                    .orderBy("order_date")

result = (
    df.withColumn(
        "first_order_date",
        F.first("order_date").over(window_spec)
    )
    .withColumn(
        "latest_order_date",
        F.last("order_date").over(window_spec)
    )
)
result.display()